# Brent · `decision_target_date` (web: "Decision target date")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and values that are not dates

Fetch `decision_target_date` for every Brent application (paged with `search_after`, sorted by `id`,
because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it;
- **not a date**: a value that doesn't parse as `DD/MM/YYYY`.

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "decision_target_date"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "decision_target_date"])
print("Brent applications:", len(df))

Brent applications: 36952


In [3]:
value = df["decision_target_date"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")
parsed = pd.to_datetime(value, format="%d/%m/%Y", errors="coerce")
not_a_date = value.notna() & ~is_blank & parsed.isna()

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "not a date": int(not_a_date.sum()),
    "valid DD/MM/YYYY date": int(parsed.notna().sum()),
})
summary["total"] = summary.sum()
display(summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)}))

print("Values that are not dates:")
df[not_a_date]

,count,share_%
null,4025,10.89
empty or blank,0,0.00
not a date,0,0.00
valid DD/MM/YYYY date,32927,89.11
total,36952,100.00


Values that are not dates:


,id,decision_target_date


### Findings · Step 1

Live API, 2026-11-10, 36,952 Brent applications:

| | Count | Share |
|---|---|---|
| null | 4,025 | 10.89% |
| empty or blank | 0 | 0% |
| not a date | 0 | 0% |
| valid `DD/MM/YYYY` date | 32,927 | 89.11% |

- **Clean format:** every filled value is a real `DD/MM/YYYY` date.
- **4,025 have no target date** (10.9%), close to the 3,928 with no `valid_date`; the
  overlap is not checked yet.

## Step 2 · Overlap with the records that have no `valid_date`

Count, with `_count` and the Brent filter, the records with no `decision_target_date`
that also have no `valid_date`, and split the records with no target date into FME Legacy
records, empty records (dropped at load) and the rest.

In [4]:
def count(filters: list[dict], not_filters: list[dict] | None = None) -> int:
    body = {"query": {"bool": {"filter": [brent_filter(), *filters], "must_not": not_filters or []}}}
    response = httpx.post(f"{BASE_URL}/applications/_count", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()["count"]


def missing(field: str) -> dict:
    return {"bool": {"must_not": [{"exists": {"field": field}}]}}


fme_legacy = {"term": {"bo_system.raw": "FME Legacy Data Import"}}
empty_record = {"bool": {"filter": [missing("status"), missing("decision"), missing("decision_date"), missing("description")]}}
no_target = missing("decision_target_date")

display(pd.Series({
    "no decision_target_date": count([no_target]),
    "  also no valid_date": count([no_target, missing("valid_date")]),
    "  has a valid_date": count([no_target], [missing("valid_date")]),
    "no valid_date but has a decision_target_date": count([missing("valid_date")], [no_target]),
}).to_frame("count"))

pd.Series({
    "no decision_target_date": count([no_target]),
    "  FME Legacy": count([no_target, fme_legacy]),
    "  empty records (dropped)": count([no_target, empty_record]),
    "  neither": count([no_target], [fme_legacy, empty_record]),
}).to_frame("count")

,count
no decision_target_date,4025
also no valid_date,3928
has a valid_date,97
no valid_date but has a decision_target_date,0


,count
no decision_target_date,4025
FME Legacy,3531
empty records (dropped),407
neither,87


### Findings · Step 2

Live API, 2026-11-10:

| | Count |
|---|---|
| no `decision_target_date` | 4,025 |
| also no `valid_date` | 3,928 |
| has a `valid_date` | 97 |
| no `valid_date` but has a target date | 0 |

| No `decision_target_date` | Count |
|---|---|
| FME Legacy records | 3,531 |
| empty records (dropped at load) | 407 |
| neither (real gaps) | 87 |

- **Every record with no `valid_date` also has no target date** (3,928 of 3,928). This fits
  a target date counted from the valid date (usually 8 or 13 weeks later); not checked.
- **All FME Legacy records and all empty records have no target date.**
- **After cleaning, 87 real records have no target date:** the 75 with no `valid_date`
  (`valid_date` Step 6) and 12 that have a valid date but no target date (the 97 with a
  valid date minus the 85 empty records that have one).

## Step 3 · Target date passed, but no decision

Records whose `decision_target_date` is before today (the day the notebook is run) but
that have no `decision_date`. Fetch `status`, `decision`, `decision_date` and the target
date for every record, then count these by `status` and by how long ago the target was.

In [5]:
FIELDS = ["status", "decision", "decision_date", "decision_target_date", "valid_date", "application_type"]
rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", *FIELDS], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

td = pd.DataFrame(rows).reindex(columns=["id", *FIELDS])
today = pd.Timestamp.today().normalize()
target = pd.to_datetime(td["decision_target_date"], format="%d/%m/%Y", errors="coerce")
overdue = td[(target < today) & td["decision_date"].isna()].copy()
overdue["days_past_target"] = (today - target[overdue.index]).dt.days
print("today:", today.date(), "| target passed, no decision_date:", len(overdue),
      "| of these with a decision value:", overdue["decision"].notna().sum())

display(overdue["status"].value_counts(dropna=False).to_frame("count"))
bins = [0, 30, 90, 365, 3 * 365, 100 * 365]
labels = ["under 1 month", "1-3 months", "3-12 months", "1-3 years", "over 3 years"]
pd.crosstab(overdue["status"].fillna("(none)"), pd.cut(overdue["days_past_target"], bins, labels=labels), margins=True)

today: 2026-11-10 | target passed, no decision_date: 199 | of these with a decision value: 14


,count
status,
Application Received,103
Application Under Consideration,55
Closed,20
Withdrawn,11
Application Invalid,10


days_past_target,under 1 month,1-3 months,3-12 months,1-3 years,over 3 years,All
status,,,,,,
Application Invalid,2,1,2,2,3,10
Application Received,70,6,1,8,18,103
Application Under Consideration,36,0,6,10,3,55
Closed,0,0,0,1,19,20
Withdrawn,0,0,0,2,9,11
All,108,7,9,23,52,199


### Findings · Step 3

Live API, run 2026-11-10: **199 records have a target date in the past and no
`decision_date`** (14 of them have a `decision` value, part of `decision_without_date`).

| `status` | Count | under 1 month | 1-3 months | 3-12 months | 1-3 years | over 3 years |
|---|---|---|---|---|---|---|
| Application Received | 103 | 70 | 6 | 1 | 8 | 18 |
| Application Under Consideration | 55 | 36 | 0 | 6 | 10 | 3 |
| Closed | 20 | 0 | 0 | 0 | 1 | 19 |
| Withdrawn | 11 | 0 | 0 | 0 | 2 | 9 |
| Application Invalid | 10 | 2 | 1 | 2 | 2 | 3 |

- **41 ended without a decision** (Closed, Withdrawn, Application Invalid): no decision is
  expected for these, so they are not overdue.
- **158 are still open** (Received or Under Consideration). 106 passed their target less
  than a month ago: normal for late decisions, and the data was last updated a few days
  before this run.
- **39 open ones passed their target over a year ago** (Received 26, Under Consideration
  13). They are most likely stale: decided or ended long ago, but the record was never
  updated. Similar to `Brent-96_1985` (open question 5).

## Step 4 · Do the values match the schema type?

The schema gives `decision_target_date` the type `date` (`DD/MM/YYYY`). JSON has no date
type, so a date should come as a string; Step 1 already showed every filled value parses.
Count the Python type of every value in `df`: `str` = JSON string, `int` / `float` = JSON
number, `NoneType` = null.

In [6]:
types = df["decision_target_date"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
display(types.value_counts().to_frame("count"))

not_str = df[~types.isin(["str", "NoneType"])]
print("values that are not strings:", len(not_str))
not_str["decision_target_date"].value_counts().head(30).to_frame("count")

,count
decision_target_date,
str,32927
NoneType,4025


values that are not strings: 0


,count
decision_target_date,


### Findings · Step 4

Live API, 2026-11-10:

| Python type | Count |
|---|---|
| `str` | 32,927 |
| null | 4,025 |
| number | 0 |

- **Every filled value is a JSON string** in `DD/MM/YYYY` (Step 1), which is how the
  schema's `date` type comes in JSON. No type mismatch.

## Step 5 · Most common values

Count identical target dates in `df` and show the 20 most common, the earliest and latest
date, and the number per year.

In [7]:
dates = pd.to_datetime(df["decision_target_date"], format="%d/%m/%Y", errors="coerce")
print("distinct dates:", dates.nunique(), "| earliest:", dates.min().date(), "| latest:", dates.max().date())
display(dates.dt.year.value_counts().sort_index().to_frame("target dates per year"))
df["decision_target_date"].value_counts().head(20).to_frame("count")

distinct dates: 2464 | earliest: 2018-01-24 | latest: 2027-05-28


,target dates per year
decision_target_date,
2018.0,3934
2019.0,4303
2020.0,3830
2021.0,4445
2022.0,3892
2023.0,3339
2024.0,3170
2025.0,3176
2026.0,2832


,count
decision_target_date,
31/08/2018,45
22/03/2022,39
11/10/2018,35
12/05/2023,35
21/10/2024,35
27/09/2018,33
09/07/2018,32
07/01/2022,32
25/11/2019,31


### Findings · Step 5

Live API, 2026-11-10: 32,927 target dates, 2,464 distinct, from 24/01/2018 to 28/05/2027.

- **No placeholder dates:** the most common date is used 45 times (31/08/2018); a busy
  day, not a filler value.
- **Target dates start in 2018:** about 3,200-4,400 a year from 2018 to 2025. Older
  applications (the legacy records) have none, so "decided on time?" can only be
  answered from 2018 on.
- **Future dates are normal:** 6 in 2027 and part of 2026 are applications not decided
  yet.

## Step 6 · Gap between `valid_date` and `decision_target_date`

For every record with both dates (from `td`, Step 3), the number of days from the valid
date to the target date. Show the most common gaps, how many are exactly 8 weeks
(56 days), 13 weeks (91 days) or 42 days, and the gaps by `application_type`.

In [8]:
valid = pd.to_datetime(td["valid_date"], format="%d/%m/%Y", errors="coerce")
target = pd.to_datetime(td["decision_target_date"], format="%d/%m/%Y", errors="coerce")
gaps = td.assign(gap_days=(target - valid).dt.days).dropna(subset=["gap_days"])
gaps["gap_days"] = gaps["gap_days"].astype(int)
print("records with both dates:", len(gaps), "| negative gaps:", int((gaps["gap_days"] < 0).sum()))

display(gaps["gap_days"].value_counts().head(15).to_frame("count").assign(
    share_pct=lambda t: (100 * t["count"] / len(gaps)).round(2)))

named = gaps["gap_days"].map({56: "56 days (8 weeks)", 91: "91 days (13 weeks)", 42: "42 days"}).fillna("other")
display(named.value_counts().to_frame("count").assign(share_pct=lambda t: (100 * t["count"] / len(gaps)).round(2)))

pd.crosstab(gaps["application_type"], named, margins=True)

records with both dates: 32927 | negative gaps: 0


,count,share_pct
gap_days,,
49,12296,37.34
42,8815,26.77
35,3415,10.37
21,736,2.24
63,371,1.13
70,327,0.99
84,217,0.66
64,199,0.60
77,183,0.56


,count,share_pct
gap_days,,
other,23994,72.87
42 days,8815,26.77
91 days (13 weeks),85,0.26
56 days (8 weeks),33,0.10


gap_days,42 days,56 days (8 weeks),91 days (13 weeks),other,All
application_type,,,,,
All Other,4034,19,56,15474,19583
Householder,4779,9,29,4594,9411
Prior Approval,2,5,0,3926,3933
All,8815,33,85,23994,32927


In [9]:
# Most common gaps per application_type, and how many gaps are whole weeks.
print("whole weeks:", int((gaps["gap_days"] % 7 == 0).sum()), "of", len(gaps))
for app_type, group in gaps.groupby("application_type"):
    top = group["gap_days"].value_counts().head(5)
    print(f"{app_type} ({len(group)}):", ", ".join(f"{d} days {c} ({100 * c / len(group):.0f}%)" for d, c in top.items()))
print("min / median / max gap:", gaps["gap_days"].min(), gaps["gap_days"].median(), gaps["gap_days"].max())

whole weeks: 27022 of 32927
All Other (19583): 49 days 9909 (51%), 42 days 4034 (21%), 21 days 706 (4%), 63 days 218 (1%), 70 days 178 (1%)
Householder (9411): 42 days 4779 (51%), 49 days 2049 (22%), 63 days 148 (2%), 70 days 131 (1%), 64 days 91 (1%)
Prior Approval (3933): 35 days 3338 (85%), 49 days 338 (9%), 48 days 91 (2%), 21 days 24 (1%), 70 days 18 (0%)
min / median / max gap: 1 49.0 2038


### Findings · Step 6

Live API, 2026-11-10, 32,927 records with both dates (no target date is before its valid
date):

| Gap (days) | Count | Share |
|---|---|---|
| 49 (7 weeks) | 12,296 | 37.3% |
| 42 (6 weeks) | 8,815 | 26.8% |
| 35 (5 weeks) | 3,415 | 10.4% |
| 21 (3 weeks) | 736 | 2.2% |
| 56 (8 weeks) | 33 | 0.1% |
| 91 (13 weeks) | 85 | 0.3% |

Most common gap by `application_type`:

| `application_type` | Most common gaps |
|---|---|
| All Other (19,583) | 49 days 51%, 42 days 21%, 21 days 4% |
| Householder (9,411) | 42 days 51%, 49 days 22% |
| Prior Approval (3,933) | 35 days 85%, 49 days 9% |

- **The 8/13-week rule does not describe this data:** only 118 records (0.4%) have a gap of
  exactly 56 or 91 days.
- **Gaps are mostly whole weeks** (27,022 of 32,927) and mostly shorter than 8 weeks:
  7 weeks for All Other, 6 weeks for Householder, 5 weeks for Prior Approval. Why Brent's
  target dates are set this way is not known from the data.
- **Wide range:** 1 to 2,038 days (median 49). Long gaps may be agreed extensions; not
  checked.